# Iterative Re-solving

In many optimization workflows you need to solve a model, tweak parameters on the native solver object, and re-solve — without rebuilding the entire linopy model. This is common in:

- Sensitivity analysis (varying objective coefficients or bounds)
- Decomposition algorithms (Benders, column generation)
- Rolling-horizon / receding-horizon schemes

Linopy supports this via two methods:

- **`Solver.resolve()`** — re-solves an existing native solver model and returns a `Result`
- **`Model.apply_result()`** — maps a `Result` back onto the linopy model's variables and constraints

Currently `resolve()` is implemented for the **HiGHS**, **Gurobi**, and **Mosek** solvers.

## Setup

We start with a simple two-variable LP.

In [ ]:
import numpy as np

from linopy import Model
from linopy.solvers import Highs

In [ ]:
m = Model()
x = m.add_variables(lower=0, upper=10, name="x")
y = m.add_variables(lower=0, upper=10, name="y")
m.add_constraints(x + y >= 8, name="demand")
m.add_constraints(x + y <= 15, name="capacity")
m.objective = 2 * x + 3 * y
m

## Initial Solve

Solve the model as usual. After solving, linopy stores the native solver object in `m.solver_model`.

In [ ]:
m.solve(solver_name="highs")
print(f"Objective: {m.objective.value}")
print(f"x = {m.solution['x'].values.item():.2f}")
print(f"y = {m.solution['y'].values.item():.2f}")

## Modify the Native Solver Model

Access the native HiGHS object and change the objective coefficients. Here we make `x` much more expensive, so the solver should prefer `y`.

In [ ]:
h = m.solver_model
n_cols = h.getNumCol()
new_costs = np.array([10.0, 1.0], dtype=float)
h.changeColsCost(n_cols, np.arange(n_cols, dtype=np.int32), new_costs)

## Re-solve and Apply

Create a solver instance, call `resolve()`, then apply the result back to the linopy model.

In [ ]:
solver = Highs()
result = solver.resolve(h, sense=m.sense)
m.apply_result(result, solver_name="highs")

print(f"Objective: {m.objective.value}")
print(f"x = {m.solution['x'].values.item():.2f}")
print(f"y = {m.solution['y'].values.item():.2f}")

Since `x` is now 10× more expensive than `y`, the solver allocates as much as possible to `y`.

## Iterating Over Parameter Sweeps

The `resolve` / `apply_result` pattern is efficient for parameter sweeps because it avoids rebuilding the model from scratch each time.

In [ ]:
import pandas as pd

cost_x_values = [1.0, 2.0, 5.0, 10.0, 20.0]
results = []

for cost_x in cost_x_values:
    h.changeColsCost(
        n_cols, np.arange(n_cols, dtype=np.int32), np.array([cost_x, 3.0], dtype=float)
    )
    result = solver.resolve(h, sense=m.sense)
    m.apply_result(result, solver_name="highs")
    results.append(
        {
            "cost_x": cost_x,
            "objective": m.objective.value,
            "x": m.solution["x"].values.item(),
            "y": m.solution["y"].values.item(),
        }
    )

pd.DataFrame(results)

## Handling Infeasible Results

When a re-solve yields an infeasible or otherwise non-optimal result, `apply_result` returns the status without setting variable solutions. You can also pass `solution=None` in a `Result` object.

In [ ]:
from linopy.constants import Result, Status, TerminationCondition

status = Status.from_termination_condition(TerminationCondition.infeasible)
result = Result(status=status, solution=None)

s, tc = m.apply_result(result)
print(f"Status: {s}, Termination: {tc}")

## API Reference

**`Solver.resolve(solver_model, sense="min")`**

Re-solves a native solver object and returns a `Result`. Implemented for `Highs`, `Gurobi`, and `Mosek`. Raises `NotImplementedError` for other solvers.

**`Model.apply_result(result, solver_name=None)`**

Applies a `Result` to the model:
1. Resets existing solutions
2. Sets status and termination condition
3. Maps primal values back to variables
4. Maps dual values back to constraints (if available)

Returns a `(status, termination_condition)` tuple.